# Multihead Self-Attention



- **What it does**: Helps models focus on important parts of data (like words in a sentence).

- **Demo**: Toy example from Notebook 12_2.



## Step 1: Create Input Data

- **Input**: A matrix `X` with 8 features (rows) and 6 items (columns).

- **Think of it**: Like 6 words, each described by 8 numbers.

- **Goal**: Transform this data using attention.



In [1]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(3)
# 6 items, each with 8 features
N, D = 6, 8
X = np.random.normal(size=(D, N))  # Random input data
print('Input X:\n', X.round(2))  # Show full input

Input X:
 [[ 1.79  0.44  0.1  -1.86 -0.28 -0.35]
 [-0.08 -0.63 -0.04 -0.48 -1.31  0.88]
 [ 0.88  1.71  0.05 -0.4  -0.55 -1.55]
 [ 0.98 -1.1  -1.19 -0.21  1.49  0.24]
 [-1.02 -0.71  0.63 -0.16 -0.77 -0.23]
 [ 0.75  1.98 -1.24 -0.63 -0.8  -2.42]
 [-0.92 -1.02  1.12 -0.13 -1.62  0.65]
 [-0.36 -1.74 -0.6  -0.59 -0.87  0.03]]


## Step 2: Setup Two Attention Heads

- **Heads**: Two mini-models (4 features each) to spot different patterns.

- **Parameters**: Weights (`omega`) and biases (`beta`) for queries, keys, values.

- **Analogy**: Like two detectives scanning data for clues.



In [2]:
# Two heads, each handling 4 features
H, H_D = 2, D // 2
np.random.seed(0)  # Consistent random numbers

# Head 1 parameters
omega_q1 = np.random.normal(size=(H_D, D))  # Query weights
omega_k1 = np.random.normal(size=(H_D, D))  # Key weights
omega_v1 = np.random.normal(size=(H_D, D))  # Value weights
beta_q1 = np.random.normal(size=(H_D, 1))   # Query bias
beta_k1 = np.random.normal(size=(H_D, 1))   # Key bias
beta_v1 = np.random.normal(size=(H_D, 1))   # Value bias

# Head 2 parameters (same structure)
omega_q2 = np.random.normal(size=(H_D, D))
omega_k2 = np.random.normal(size=(H_D, D))
omega_v2 = np.random.normal(size=(H_D, D))
beta_q2 = np.random.normal(size=(H_D, 1))
beta_k2 = np.random.normal(size=(H_D, 1))
beta_v2 = np.random.normal(size=(H_D, 1))

# Final transformation matrix
omega_c = np.random.normal(size=(D, D))

## Step 3: Softmax for Attention Weights

- **What**: Turns scores into weights (0 to 1) for focusing on data.

- **How**: Exponentiates scores, normalizes by column sums.

- **Why**: Highlights important items in the input.



In [3]:
def softmax_cols(data_in):
    exp_values = np.exp(data_in)  # Exponentiate scores
    denom = np.sum(exp_values, axis=0)  # Sum per column
    return exp_values / denom  # Normalize to weights

## Step 4: Run Multihead Attention

- **Process**:

  - Each head computes queries, keys, values.

  - Scores measure item similarity, scaled to avoid huge numbers.

  - Softmax weights focus on key items, combine heads.

- **Insight**: Like a hacker prioritizing critical data patterns.



In [4]:
def multihead_scaled_self_attention(X, omega_v1, omega_q1, omega_k1, beta_v1, beta_q1, beta_k1,
                                   omega_v2, omega_q2, omega_k2, beta_v2, beta_q2, beta_k2, omega_c):
    H_D = omega_q1.shape[0]  # Features per head (4)

    # Head 1: Queries, Keys, Values
    Q1 = omega_q1 @ X + beta_q1  # What to look for
    K1 = omega_k1 @ X + beta_k1  # What to compare
    V1 = omega_v1 @ X + beta_v1  # What to output
    scores1 = (Q1.T @ K1) / np.sqrt(H_D)  # Similarity scores, scaled
    attention_weights1 = softmax_cols(scores1.T)  # Focus weights
    head1 = V1 @ attention_weights1  # Weighted output

    # Head 2: Same steps
    Q2 = omega_q2 @ X + beta_q2
    K2 = omega_k2 @ X + beta_k2
    V2 = omega_v2 @ X + beta_v2
    scores2 = (Q2.T @ K2) / np.sqrt(H_D)
    attention_weights2 = softmax_cols(scores2.T)
    head2 = V2 @ attention_weights2

    # Combine heads and transform
    concat_heads = np.vstack((head1, head2))  # Stack outputs
    X_prime = omega_c @ concat_heads  # Final transformation

    return X_prime

# Run it
X_prime = multihead_scaled_self_attention(X, omega_v1, omega_q1, omega_k1, beta_v1, beta_q1, beta_k1,
                                         omega_v2, omega_q2, omega_k2, beta_v2, beta_q2, beta_k2, omega_c)
print('Output X_prime:\n', X_prime.round(3))

Output X_prime:
 [[-21.207  -5.373 -20.933  -9.179 -11.319 -17.812]
 [ -1.995   7.906 -10.516   3.452   9.863  -7.24 ]
 [  5.479   1.115   9.244   0.453   5.656   7.089]
 [ -7.413  -7.416   0.363  -5.573  -6.736  -0.848]
 [-11.261  -9.937  -4.848  -8.915 -13.378  -5.761]
 [  3.548  10.036  -2.244   1.604  12.113  -2.557]
 [  4.888  -5.814   2.407   3.228  -4.232   3.71 ]
 [  1.248  18.894  -6.409   3.224  19.717  -5.629]]


## Wrap-Up

- **Result**: `X_prime` captures key input relationships.

- **Why it matters**: Powers transformers for tasks like translation, text generation, and more.



